# 04 — ARIMA & SARIMA

**Mục tiêu:** Xây dựng và so sánh mô hình ARIMA và SARIMA để dự báo doanh số,
đánh giá bằng RMSLE, chẩn đoán bằng Ljung-Box.

**Kết quả từ các Task trước:**
- **Task 2 (Decomposition):** MSTL phân rã → Trend + Seasonal (tuần/tháng/quý/năm) + Residual.
- **Task 3 (QS Test):** Xác nhận mùa vụ tuần (m=7, p≈0), quý (m=91, p≈0), năm (m=365, p=0.021) có ý nghĩa. Tháng (m=30, p=0.052) KHÔNG có ý nghĩa.

**Workflow:**
1. Load dữ liệu `train_split.csv`, `test_split.csv` + kết quả QS Test
2. Kiểm định tính dừng (ADF + KPSS)
3. Xác định m cho SARIMA từ QS test — xây phương án cho từng m có ý nghĩa
4. `auto_arima` tìm order tối ưu cho ARIMA và từng SARIMA(m)
5. Fit chính thức bằng `statsmodels.SARIMAX`, kiểm tra hội tụ
6. Ljung-Box (loại đúng `loglikelihood_burn`)
7. Dự báo trên test, tính RMSLE, so sánh Baseline / ARIMA / tất cả SARIMA(m)
8. Lưu kết quả cho Task 5

In [ ]:
import sys
import os

# Thêm thư mục gốc project vào sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima import auto_arima

from src.arima_sarima import (
    check_stationarity,
    fit_sarimax,
    check_ljungbox,
    rmsle
)

print('✓ Import thành công tất cả modules!')

## Bước 2 — Load dữ liệu và kết quả QS Test (Task 3)

In [ ]:
# Load train/test splits
train = pd.read_csv(
    os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_split.csv'),
    parse_dates=['date'], index_col='date'
).asfreq('D')

test = pd.read_csv(
    os.path.join(PROJECT_ROOT, 'data', 'processed', 'test_split.csv'),
    parse_dates=['date'], index_col='date'
).asfreq('D')

# Sử dụng sales_log (log-transformed)
train_log = train['sales_log']

print(f'Train: {len(train)} ngày  ({train.index.min().date()} → {train.index.max().date()})')
print(f'Test : {len(test)} ngày  ({test.index.min().date()} → {test.index.max().date()})')
print(f'\nCột train: {list(train.columns)}')
train.head()

In [ ]:
# Load kết quả QS test từ Task 3
qs_results = pd.read_csv(
    os.path.join(PROJECT_ROOT, 'results', 'metrics', 'seasonality_qs_test_summary.csv')
)
print('Kết quả QS Test (Task 3):')
print(qs_results.to_string(index=False))

## Bước 3 — Xác định m dùng cho SARIMA

**Quy tắc lựa chọn (khách quan, không chọn cảm tính):**
- Nếu **chỉ 1** tần suất có ý nghĩa → dùng đúng m đó.
- Nếu **nhiều hơn 1** → xây SARIMA riêng cho **từng m**, so sánh RMSLE.
- Nếu **không có** tần suất nào → chỉ chạy ARIMA.
- **m ≥ 365** → ghi chú giới hạn kỹ thuật (quá lớn cho SARIMA).

In [ ]:
# Trích các tần suất có ý nghĩa thống kê
significant_freqs = qs_results[qs_results['Kết luận'] == 'CÓ mùa vụ']
print(f"Các tần suất có ý nghĩa thống kê:")
print(significant_freqs[['Tần suất', 'm', 'QS_stat', 'p_value']].to_string(index=False))

# Xác định danh sách m sẽ xây SARIMA
sarima_m_list = []
skipped_m = []

for _, row in significant_freqs.iterrows():
    m = int(row['m'])
    if m >= 365:
        skipped_m.append((row['Tần suất'], m))
        print(f"\n⚠ {row['Tần suất']} (m={m}): BỎ QUA — m quá lớn, SARIMA không khả thi kỹ thuật.")
        print(f"  Lý do: Với m={m}, SARIMA cần ước lượng seasonal component với chu kỳ {m} ngày,")
        print(f"  đòi hỏi bộ nhớ lớn và thời gian fit rất lâu. Đây là giới hạn của mô hình SARIMA.")
    else:
        sarima_m_list.append(m)
        print(f"\n✓ {row['Tần suất']} (m={m}): Sẽ xây SARIMA(m={m})")

print(f"\n→ Danh sách m cho SARIMA: {sarima_m_list}")
if skipped_m:
    print(f"→ Bỏ qua (giới hạn kỹ thuật): {skipped_m}")

## Bước 4 — Kiểm định tính dừng (ADF + KPSS)

In [ ]:
# Kiểm định trên train_log
stationarity = check_stationarity(train_log)

print('='*65)
print('  KIỂM ĐỊNH TÍNH DỪNG — train_log (log-transformed sales)')
print('='*65)

print('\n— ADF Test (H₀: có unit root = KHÔNG dừng) —')
print(f'  ADF Statistic : {stationarity["adf_stat"]:.4f}')
print(f'  p-value       : {stationarity["adf_pvalue"]:.6f}')
print(f'  Số lags       : {stationarity["adf_lags"]}')
print(f'  Kết luận      : {"✓ DỪNG (bác bỏ H₀)" if stationarity["adf_stationary"] else "✗ CHƯA DỪNG"}')
for key, val in stationarity['adf_critical'].items():
    print(f'  Critical ({key}): {val:.4f}')

print('\n— KPSS Test (H₀: chuỗi DỪNG) —')
print(f'  KPSS Statistic: {stationarity["kpss_stat"]:.4f}')
print(f'  p-value       : {stationarity["kpss_pvalue"]:.4f}')
print(f'  Kết luận      : {"✓ DỪNG (không bác bỏ H₀)" if stationarity["kpss_stationary"] else "✗ CHƯA DỪNG (bác bỏ H₀)"}')
for key, val in stationarity['kpss_critical'].items():
    print(f'  Critical ({key}): {val:.4f}')

# Tổng kết
if stationarity['adf_stationary'] and stationarity['kpss_stationary']:
    d_suggestion = 0
    print('\n→ Cả ADF và KPSS đều xác nhận DỪNG → d = 0')
elif not stationarity['adf_stationary'] and not stationarity['kpss_stationary']:
    d_suggestion = 1
    print('\n→ Cả ADF và KPSS đều cho thấy CHƯA DỪNG → d ≥ 1')
else:
    d_suggestion = 0
    print('\n→ ADF và KPSS cho kết quả trái ngược — để auto_arima quyết định d')

In [ ]:
# Biểu đồ ACF/PACF trên train_log
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train_log.dropna(), lags=50, ax=axes[0], title='ACF — train_log')
plot_pacf(train_log.dropna(), lags=50, ax=axes[1], title='PACF — train_log', method='ywm')
plt.suptitle('ACF & PACF trên sales_log', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'acf_pacf_log.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Đã lưu biểu đồ ACF/PACF')

## Bước 5 — auto_arima: tìm bậc tối ưu cho ARIMA và từng SARIMA(m)

In [ ]:
# --- ARIMA (non-seasonal) ---
print('='*65)
print('  auto_arima — ARIMA (seasonal=False)')
print('='*65)

auto_arima_model = auto_arima(
    train_log,
    seasonal=False,
    d=d_suggestion,
    trend='c',
    stepwise=True,
    suppress_warnings=True,
    trace=True
)

print(f'\n→ Best ARIMA order: {auto_arima_model.order}')
print(f'  AIC: {auto_arima_model.aic():.2f}')
print(auto_arima_model.summary())

In [ ]:
# --- SARIMA cho từng m có ý nghĩa ---
sarima_candidates = {}

for m in sarima_m_list:
    print('\n' + '='*65)
    print(f'  auto_arima — SARIMA (m={m})')
    print('='*65)
    
    try:
        auto_model = auto_arima(
            train_log,
            seasonal=True,
            m=m,
            d=d_suggestion,
            D=None,     # để auto_arima tự chọn D
            trend='c',
            stepwise=True,
            suppress_warnings=True,
            trace=True
        )
        sarima_candidates[m] = auto_model
        print(f'\n→ Best SARIMA(m={m}) order: {auto_model.order} x {auto_model.seasonal_order}')
        print(f'  AIC: {auto_model.aic():.2f}')
    except Exception as e:
        print(f'\n✗ auto_arima thất bại cho m={m}: {e}')

print(f'\n→ Tổng số phương án SARIMA: {len(sarima_candidates)}')

## Bước 6 — Fit chính thức bằng `statsmodels.SARIMAX`

Yêu cầu: mỗi model phải **converged = True**.

In [ ]:
# Fit ARIMA chính thức
print('='*65)
print('  Fit ARIMA chính thức')
print('='*65)

model_arima = fit_sarimax(
    train_log,
    order=auto_arima_model.order,
    seasonal_order=None,
    trend='c'
)
print(f'✓ ARIMA{auto_arima_model.order} — converged = True')
print(f'  AIC: {model_arima.aic:.2f}  BIC: {model_arima.bic:.2f}')
print(model_arima.summary())

In [ ]:
# Fit SARIMA chính thức cho từng m
fitted_sarima_models = {}

for m, auto_model in sarima_candidates.items():
    print(f'\n{"="*65}')
    print(f'  Fit SARIMA (m={m}) chính thức')
    print(f'{"="*65}')
    
    try:
        mdl = fit_sarimax(
            train_log,
            order=auto_model.order,
            seasonal_order=auto_model.seasonal_order,
            trend='c'
        )
        fitted_sarima_models[m] = mdl
        print(f'✓ SARIMA{auto_model.order}x{auto_model.seasonal_order} — converged = True')
        print(f'  AIC: {mdl.aic:.2f}  BIC: {mdl.bic:.2f}')
    except AssertionError as e:
        print(f'✗ SARIMA(m={m}): {e}')
    except Exception as e:
        print(f'✗ SARIMA(m={m}) lỗi: {e}')

print(f'\n→ Models đã fit thành công: ARIMA + {len(fitted_sarima_models)} SARIMA')

## Bước 7 — Kiểm định Ljung-Box (loại đúng `loglikelihood_burn`)

- **H₀:** Phần dư là white noise (không tự tương quan)
- **H₁:** Phần dư có tự tương quan
- Nếu p-value > 0.05 → không bác bỏ H₀ → mô hình phù hợp
- Lags: [7, 14, 21, 28, 35, 42, 49]

In [ ]:
# Ljung-Box cho ARIMA
print('='*65)
print(f'  Ljung-Box — ARIMA{auto_arima_model.order}')
print(f'  loglikelihood_burn = {model_arima.loglikelihood_burn}')
print('='*65)

lb_arima = check_ljungbox(model_arima)
print(lb_arima.to_string())
print()
for lag in [7, 14, 28, 49]:
    if lag in lb_arima.index:
        p = lb_arima.loc[lag, 'lb_pvalue']
        print(f'  Lag {lag:2d}: p = {p:.6f} → {"✓ White noise" if p > 0.05 else "✗ Tự tương quan"}')

In [ ]:
# Ljung-Box cho từng SARIMA(m)
lb_sarima_by_m = {}

for m, mdl in fitted_sarima_models.items():
    print(f'\n{"="*65}')
    order_str = f'{sarima_candidates[m].order}x{sarima_candidates[m].seasonal_order}'
    print(f'  Ljung-Box — SARIMA {order_str}')
    print(f'  loglikelihood_burn = {mdl.loglikelihood_burn}')
    print(f'{"="*65}')
    
    lb = check_ljungbox(mdl)
    lb_sarima_by_m[m] = lb
    print(lb.to_string())
    print()
    for lag in [7, 14, 28, 49]:
        if lag in lb.index:
            p = lb.loc[lag, 'lb_pvalue']
            print(f'  Lag {lag:2d}: p = {p:.6f} → {"✓ White noise" if p > 0.05 else "✗ Tự tương quan"}')

In [ ]:
# Diagnostic plots
fig = model_arima.plot_diagnostics(figsize=(14, 10))
fig.suptitle(f'Diagnostic — ARIMA{auto_arima_model.order}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'arima_diagnostics.png'),
            dpi=150, bbox_inches='tight')
plt.show()

for m, mdl in fitted_sarima_models.items():
    fig = mdl.plot_diagnostics(figsize=(14, 10))
    order_str = f'{sarima_candidates[m].order}x{sarima_candidates[m].seasonal_order}'
    fig.suptitle(f'Diagnostic — SARIMA {order_str}',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', f'sarima_m{m}_diagnostics.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

print('✓ Đã lưu biểu đồ diagnostic')

## Bước 8 — Dự báo, tính RMSLE, so sánh TẤT CẢ phương án

Dự báo trên `sales_log` → `expm1()` chuyển về sales gốc → tính RMSLE.

In [ ]:
n_test = len(test)

# --- ARIMA forecast ---
forecast_arima_log = model_arima.forecast(steps=n_test)
forecast_arima = np.maximum(np.expm1(forecast_arima_log), 0)

# --- SARIMA forecasts ---
forecast_sarima = {}
for m, mdl in fitted_sarima_models.items():
    fc_log = mdl.forecast(steps=n_test)
    forecast_sarima[m] = np.maximum(np.expm1(fc_log), 0)

# --- Baseline ---
baseline_score = pd.read_csv(
    os.path.join(PROJECT_ROOT, 'results', 'metrics', 'baseline_rmsle.csv')
)['RMSLE'].iloc[0]

# --- Tính RMSLE ---
y_true = test['sales'].values

results_list = []
results_list.append({
    'Model': 'Baseline (Seasonal Naive)',
    'm': '-',
    'RMSLE': baseline_score
})
results_list.append({
    'Model': f'ARIMA{auto_arima_model.order}',
    'm': '-',
    'RMSLE': rmsle(y_true, forecast_arima.values)
})
for m in sorted(fitted_sarima_models.keys()):
    order_str = f'{sarima_candidates[m].order}x{sarima_candidates[m].seasonal_order}'
    results_list.append({
        'Model': f'SARIMA {order_str}',
        'm': m,
        'RMSLE': rmsle(y_true, forecast_sarima[m].values)
    })

comparison = pd.DataFrame(results_list).sort_values('RMSLE').reset_index(drop=True)
comparison['Rank'] = range(1, len(comparison) + 1)

print('='*70)
print('  SO SÁNH RMSLE — TẤT CẢ PHƯƠNG ÁN')
print('='*70)
print(comparison.to_string(index=False))

best = comparison.iloc[0]
print(f'\n→ Mô hình tốt nhất: {best["Model"]} (RMSLE = {best["RMSLE"]:.6f})')

# Improvement vs Baseline
for _, row in comparison.iterrows():
    if row['Model'] != 'Baseline (Seasonal Naive)':
        imp = ((baseline_score - row['RMSLE']) / baseline_score) * 100
        direction = '↓ tốt hơn' if imp > 0 else '↑ tệ hơn'
        print(f'  {row["Model"]:40s} vs Baseline: {direction} {abs(imp):.2f}%')

## Bước 9 — Lưu model tốt nhất và dự báo chi tiết

In [ ]:
# 9a. Lưu bảng so sánh RMSLE
output_comparison = os.path.join(PROJECT_ROOT, 'results', 'metrics', 'model_comparison_rmsle.csv')
os.makedirs(os.path.dirname(output_comparison), exist_ok=True)
comparison.to_csv(output_comparison, index=False)
print(f'✓ Lưu: {output_comparison}')

# 9b. Lưu dự báo chi tiết (forecast_test_2017.csv)
forecast_df = pd.DataFrame({
    'date': test.index,
    'actual': y_true,
    'arima_pred': forecast_arima.values
})
for m in sorted(fitted_sarima_models.keys()):
    forecast_df[f'sarima_m{m}_pred'] = forecast_sarima[m].values
output_forecast = os.path.join(PROJECT_ROOT, 'results', 'metrics', 'forecast_test_2017.csv')
forecast_df.to_csv(output_forecast, index=False)
print(f'✓ Lưu: {output_forecast}')

# 9c. Lưu models (.pkl)
models_dir = os.path.join(PROJECT_ROOT, 'results', 'models')
os.makedirs(models_dir, exist_ok=True)

model_arima.save(os.path.join(models_dir, 'arima_model.pkl'))
print(f'✓ Lưu: results/models/arima_model.pkl')

# Lưu SARIMA tốt nhất
sarima_rows = comparison[comparison['Model'].str.contains('SARIMA')]
if len(sarima_rows) > 0:
    best_sarima_m = int(sarima_rows.iloc[0]['m'])
    best_sarima_model = fitted_sarima_models[best_sarima_m]
    best_sarima_model.save(os.path.join(models_dir, 'sarima_best_model.pkl'))
    print(f'✓ Lưu: results/models/sarima_best_model.pkl (m={best_sarima_m})')

print('\n✓ Tất cả model và kết quả đã được lưu!')

## Bước 10 — Trực quan hóa: Actual vs ARIMA vs từng SARIMA(m)

In [ ]:
n_models = 1 + len(fitted_sarima_models)  # ARIMA + SARIMAs
colors = ['#E91E63', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# --- Plot 1: Toàn cảnh ---
ax1 = axes[0]
ax1.plot(train.index, train['sales'], color='#2196F3', alpha=0.4,
         linewidth=0.6, label='Train')
ax1.plot(test.index, y_true, color='#333333', linewidth=1.3,
         label='Test (Actual)')

arima_rmsle = rmsle(y_true, forecast_arima.values)
ax1.plot(test.index, forecast_arima.values, color=colors[0],
         linewidth=1.2, linestyle='--',
         label=f'ARIMA{auto_arima_model.order} (RMSLE={arima_rmsle:.4f})')

for i, m in enumerate(sorted(fitted_sarima_models.keys())):
    order_str = f'{sarima_candidates[m].order}x{sarima_candidates[m].seasonal_order}'
    s_rmsle = rmsle(y_true, forecast_sarima[m].values)
    ax1.plot(test.index, forecast_sarima[m].values,
             color=colors[(i+1) % len(colors)],
             linewidth=1.2, linestyle='-.',
             label=f'SARIMA {order_str} (RMSLE={s_rmsle:.4f})')

ax1.axvline(x=pd.Timestamp('2017-04-01'), color='red', linestyle=':',
            linewidth=1.5, alpha=0.7)
ax1.set_title('Forecast Comparison — ARIMA vs SARIMA(m) vs Actual',
              fontsize=14, fontweight='bold')
ax1.set_ylabel('Sales')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Plot 2: Zoom Test period ---
ax2 = axes[1]
ax2.plot(test.index, y_true, color='#333333', linewidth=1.5,
         marker='o', markersize=1.5, label='Actual')
ax2.plot(test.index, forecast_arima.values, color=colors[0],
         linewidth=1.2, linestyle='--', label=f'ARIMA')

for i, m in enumerate(sorted(fitted_sarima_models.keys())):
    ax2.plot(test.index, forecast_sarima[m].values,
             color=colors[(i+1) % len(colors)],
             linewidth=1.2, linestyle='-.',
             label=f'SARIMA(m={m})')

ax2.set_title('Zoom: Test Period', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Sales')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(PROJECT_ROOT, 'results', 'figures', 'forecast_comparison.png')
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Đã lưu: {fig_path}')

## Kết luận

### Tóm tắt:
- **m cho SARIMA** được lấy trực tiếp từ kết quả QS test (Task 3), không tự chọn.
- **Nếu nhiều m có ý nghĩa** → xây SARIMA riêng cho từng m, so sánh bằng RMSLE.
- **m = 365** (năm): ghi nhận có ý nghĩa thống kê nhưng **không khả thi kỹ thuật** cho SARIMA.
- Kiểm định phần dư bằng **Ljung-Box** (đã loại đúng `loglikelihood_burn` warmup).
- Tất cả model đều **converged = True**.

### Sản phẩm đầu ra:
| Loại | Đường dẫn |
|------|-----------|
| Notebook | `notebooks/04_arima_sarima.ipynb` |
| Model ARIMA | `results/models/arima_model.pkl` |
| Model SARIMA tốt nhất | `results/models/sarima_best_model.pkl` |
| Dự báo chi tiết | `results/metrics/forecast_test_2017.csv` |
| Bảng so sánh RMSLE | `results/metrics/model_comparison_rmsle.csv` |
| Biểu đồ | `results/figures/forecast_comparison.png` |